# NEPSE-Impact-500: Qwen3-8B Unsloth QLoRA

This notebook evaluates base Qwen3-8B zero-shot and three-shot, then trains a
QLoRA adapter with Unsloth on the same frozen balanced manifest used by
XLM-R. It produces compact deterministic JSON and evaluates relevance, event
type, direction, sector, symbol, and evidence selection.

In [ ]:
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install -q 'datasets>=3.2,<4' 'trl>=0.15,<1'   'matplotlib>=3.9,<4' 'seaborn>=0.13,<1' 'tqdm>=4.66,<5'   'openai>=1.50,<2'

In [ ]:
!pip install -q --upgrade --force-reinstall "numpy>=1.26.4,<1.28" "urllib3<=2.5.0"
!pip check

## 1. Verify the GPU and load the frozen corpus

In [ ]:
import torch
assert torch.cuda.is_available(), "Use a Colab or Kaggle GPU runtime."
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT = Path(os.getenv("MARKET_GYAN_PROJECT", "/content/marketGyan"))
DATA = PROJECT / "data/processed"
SPLITS = DATA / "splits"
OUTPUTS = PROJECT / "outputs"
sys.path.insert(0, str(PROJECT))

from market_gyan.dataset import (
    balanced_group_split,
    compact_qwen_label,
    dataset_readiness,
    read_jsonl,
    split_manifest,
    validate_dataset,
    write_jsonl,
)

gold_path = DATA / "nepse-impact-500.jsonl"
rows = read_jsonl(gold_path)
issues = validate_dataset(rows)
gate = dataset_readiness(rows)
print(json.dumps(gate, indent=2, ensure_ascii=False))
assert not issues, issues[:3]
assert gate["ready"], gate["errors"]

SPLITS.mkdir(parents=True, exist_ok=True)
manifest_path = SPLITS / "manifest.json"
if not manifest_path.exists():
    frozen = balanced_group_split(rows)
    for name, values in frozen.items():
        write_jsonl(SPLITS / f"{name}.jsonl", values)
    manifest_path.write_text(
        json.dumps(
            split_manifest(frozen, strategy="balanced"),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert {item["id"] for item in manifest["assignments"]} == {row["id"] for row in rows}
group_splits = {}
for item in manifest["assignments"]:
    previous = group_splits.setdefault(item["duplicateGroupId"], item["split"])
    assert previous == item["split"], "Near-duplicate group crosses split boundaries"
print(manifest["counts"], manifest["sha256"])

In [ ]:
train_rows = read_jsonl(SPLITS / "train.jsonl")
validation_rows = read_jsonl(SPLITS / "validation.jsonl")
test_rows = read_jsonl(SPLITS / "test.jsonl")
print(len(train_rows), len(validation_rows), len(test_rows))

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def plot_counter(axis, counter, title, color, rotate=False):
    items = sorted(counter.items(), key=lambda item: str(item[0]))
    if not items:
        axis.text(0.5, 0.5, "No records", ha="center", va="center")
        axis.set_xticks([])
    else:
        labels, values = zip(*items)
        bars = axis.bar(list(labels), list(values), color=color)
        axis.bar_label(bars, padding=2, fontsize=8)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.2)
    if rotate:
        axis.tick_params(axis="x", rotation=65)

relevance = Counter(row["gold"]["relevance"] for row in rows)
languages = Counter(row["gold"]["language"] for row in rows)
events = Counter(row["gold"]["eventType"] for row in rows)
directions = Counter(
    row["gold"]["impactDirection"]
    for row in rows if row["gold"]["relevance"] != "not_relevant"
)

OUTPUTS.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_counter(axes[0, 0], relevance, "Relevance", "#2563eb")
plot_counter(axes[0, 1], languages, "Language", "#0f766e")
plot_counter(axes[1, 0], events, "Event type", "#7c3aed", rotate=True)
plot_counter(axes[1, 1], directions, "Relevant-record direction", "#dc2626")
plt.tight_layout()
plt.savefig(OUTPUTS / "nepse_impact_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 2. Load Qwen3 through Unsloth and define the structured prompt

In [ ]:
from unsloth import FastLanguageModel
from market_gyan.structured_output import compact_qwen_response_format

MODEL_NAME = "unsloth/Qwen3-8B"
MAX_SEQ_LENGTH = 1536
MAX_GENERATION_TOKENS = 192
MAX_PROMPT_TOKENS = MAX_SEQ_LENGTH - MAX_GENERATION_TOKENS
use_bf16 = torch.cuda.is_bf16_supported()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_SEQ_LENGTH
output_dir = OUTPUTS / "marketgyan-qwen3-8b-unsloth-qlora"
USE_VLLM_CONSTRAINED = (
    os.getenv("MARKET_GYAN_USE_VLLM_CONSTRAINED", "false").lower()
    in {"1", "true", "yes"}
)
VLLM_BASE_URL = os.getenv("MARKET_GYAN_VLLM_BASE_URL", "http://127.0.0.1:8000/v1")
VLLM_API_KEY = os.getenv("MARKET_GYAN_VLLM_API_KEY", "local")
VLLM_MODEL = os.getenv("MARKET_GYAN_VLLM_MODEL", "marketgyan-qwen3-8b")

COMPACT_SCHEMA_INSTRUCTIONS = (
    "Return only valid compact JSON. Do not use markdown. Do not explain. "
    "Allowed relevance: direct, indirect, not_relevant. "
    "Allowed eventType: market_trading, earnings, capital_action, governance, "
    "project_operations, credit_financing, regulation, monetary_liquidity, "
    "fiscal_macroeconomic, sector_industry, other, not_applicable. "
    "Allowed impactScope: company, sector, market, none. "
    "Allowed impactDirection: bullish, bearish, neutral, uncertain, not_applicable. "
    "Allowed impactHorizon: immediate, short_term, medium_term, not_applicable. "
    "Allowed impactMechanism: earnings_cash_flow, ownership_supply, "
    "financing_liquidity, regulation, demand_revenue, operations_capacity, "
    "valuation_sentiment, market_flow, uncertain, none. "
    "Allowed confidenceBand: low, medium, high. "
    "Required keys: relevance, eventType, impactScope, impactDirection, "
    "impactHorizon, impactMechanism, sectors, symbols, confidenceBand, "
    "evidenceSentenceIds. For not_relevant use eventType=not_applicable, "
    "impactScope=none, impactDirection=not_applicable, "
    "impactHorizon=not_applicable, impactMechanism=none, sectors=[], symbols=[]. "
    "Use only numbered evidenceSentenceIds from the source."
)

def prompt_for(row):
    numbered = "\n".join(
        f"[{sentence['id']}] {sentence['text']}"
        for sentence in row["sentences"]
    )
    return (
        f"{COMPACT_SCHEMA_INSTRUCTIONS}\n"
        f"Title: {row['title']}\n{numbered}"
    )

## 3. Base-model zero-shot and three-shot evaluation

In [ ]:
import copy
from tqdm.auto import tqdm

def chat_messages_for(row, demonstrations=None):
    messages = []
    for example in demonstrations or []:
        messages += [
            {"role": "user", "content": prompt_for(example)},
            {
                "role": "assistant",
                "content": json.dumps(
                    compact_qwen_label(example["gold"]),
                    ensure_ascii=False,
                    sort_keys=True,
                ),
            },
        ]
    messages.append({"role": "user", "content": prompt_for(row)})
    return messages

def tokenize_generation_prompt(text):
    previous_side = tokenizer.truncation_side
    tokenizer.truncation_side = "left"
    try:
        encoded = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_PROMPT_TOKENS,
        )
    finally:
        tokenizer.truncation_side = previous_side
    return encoded.to(model.device)

def generate_json(row, demonstrations=None):
    messages = chat_messages_for(row, demonstrations)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    # Prefixing the first JSON brace prevents Qwen from starting with Markdown
    # bullets while keeping official scoring strict.
    text = text + "{"
    inputs = tokenize_generation_prompt(text)
    generation_config = copy.deepcopy(model.generation_config)
    generation_config.max_length = None
    generation_config.max_new_tokens = None
    with torch.no_grad():
        output = model.generate(
            **inputs,
            generation_config=generation_config,
            max_new_tokens=MAX_GENERATION_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    raw = "{" + raw
    if raw.startswith("{{"):
        raw = raw[1:]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}")
        if start >= 0 and end > start:
            try:
                return json.loads(raw[start:end + 1])
            except json.JSONDecodeError:
                pass
        return {"raw": raw}

def generate_json_constrained(row, demonstrations=None):
    from openai import OpenAI

    client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)
    response = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=chat_messages_for(row, demonstrations),
        temperature=0,
        max_tokens=MAX_GENERATION_TOKENS,
        response_format=compact_qwen_response_format(row.get("sentences", [])),
    )
    raw = response.choices[0].message.content or ""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"raw": raw}

three_shot_examples = [
    next(row for row in train_rows if row["gold"]["relevance"] == value)
    for value in ("direct", "indirect", "not_relevant")
]
zero_shot = []
for row in tqdm(test_rows, desc="Qwen zero-shot", unit="doc"):
    zero_shot.append({"id": row["id"], "prediction": generate_json(row)})

three_shot = []
for row in tqdm(test_rows, desc="Qwen three-shot", unit="doc"):
    three_shot.append({
        "id": row["id"],
        "prediction": generate_json(row, three_shot_examples),
    })
output_dir.mkdir(parents=True, exist_ok=True)
write_jsonl(output_dir / "qwen_base_zero_shot.jsonl", zero_shot)
write_jsonl(output_dir / "qwen_base_three_shot.jsonl", three_shot)

constrained_zero_shot = []
constrained_three_shot = []
if USE_VLLM_CONSTRAINED:
    print(f"Running constrained Qwen through {VLLM_BASE_URL} model={VLLM_MODEL}")
    for row in tqdm(test_rows, desc="Qwen constrained zero-shot", unit="doc"):
        constrained_zero_shot.append({
            "id": row["id"],
            "prediction": generate_json_constrained(row),
        })
    for row in tqdm(test_rows, desc="Qwen constrained three-shot", unit="doc"):
        constrained_three_shot.append({
            "id": row["id"],
            "prediction": generate_json_constrained(row, three_shot_examples),
        })
    write_jsonl(
        output_dir / "qwen_vllm_constrained_zero_shot.jsonl",
        constrained_zero_shot,
    )
    write_jsonl(
        output_dir / "qwen_vllm_constrained_three_shot.jsonl",
        constrained_three_shot,
    )
else:
    print(
        "Skipping vLLM constrained decoding. Set "
        "MARKET_GYAN_USE_VLLM_CONSTRAINED=true after serving Qwen through "
        "an OpenAI-compatible vLLM endpoint."
    )

## 4. Format compact gold JSON for supervised fine-tuning

In [ ]:
from datasets import Dataset

def chat_messages(row):
    return [
        {"role": "user", "content": prompt_for(row)},
        {
            "role": "assistant",
            "content": json.dumps(
                compact_qwen_label(row["gold"]),
                ensure_ascii=False,
                sort_keys=True,
            ),
        },
    ]

def format_training_text(row):
    return tokenizer.apply_chat_template(
        chat_messages(row),
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

def oversample_training_rows(values):
    selected = list(values)
    selected.extend(
        row for row in values
        if row["gold"]["language"] == "ne"
    )
    selected.extend(
        row for row in values
        if row["gold"]["relevance"] in {"indirect", "not_relevant"}
    )
    selected.extend(
        row for row in values
        if (
            row["gold"]["relevance"] in {"indirect", "not_relevant"}
            and row["gold"]["language"] == "ne"
        )
    )
    return selected

def make_dataset(values, include_hard_negatives=True, oversample=False):
    selected = values if include_hard_negatives else [
        row for row in values if row["gold"]["relevance"] != "not_relevant"
    ]
    if oversample:
        selected = oversample_training_rows(selected)
    return Dataset.from_list([
        {"text": format_training_text(row)}
        for row in selected
    ])

train_data = make_dataset(train_rows, oversample=True)
validation_data = make_dataset(validation_rows)
print(len(train_data), len(validation_data))
print(train_data[0]["text"][:600])

## 5. Attach Unsloth LoRA and train or resume

In [ ]:
from transformers import set_seed
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
set_seed(42)
output_dir = OUTPUTS / "marketgyan-qwen3-8b-unsloth-qlora"
updates_per_epoch = max(1, (len(train_data) + 15) // 16)
print(
    f"Training Unsloth QLoRA: train={len(train_data)}, "
    f"validation={len(validation_data)}, approx_steps={updates_per_epoch * 5}"
)
arguments = SFTConfig(
    output_dir=str(output_dir),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    num_train_epochs=5,
    warmup_ratio=0.05,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",
    report_to=[],
    seed=42,
    disable_tqdm=False,
)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=arguments,
    train_dataset=train_data,
    eval_dataset=validation_data,
)

# Mask user tokens so the adapter learns only the reviewed JSON response.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
import shutil
if output_dir.exists():
    for checkpoint in output_dir.glob("checkpoint-*"):
        shutil.rmtree(checkpoint, ignore_errors=True)
trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
for checkpoint in output_dir.glob("checkpoint-*"):
    shutil.rmtree(checkpoint, ignore_errors=True)

## 6. Smoke-test strict JSON before full held-out generation

In [ ]:
from market_gyan.metrics import benchmark_predictions

FastLanguageModel.for_inference(model)
model.eval()
ALLOW_FAILED_QWEN_SMOKE = (
    os.getenv("MARKET_GYAN_ALLOW_FAILED_QWEN_SMOKE", "false").lower()
    in {"1", "true", "yes"}
)

def smoke_rows_for_generation(rows, limit=10):
    selected = []
    for language in ("ne", "en"):
        selected.extend([
            row for row in rows
            if row["gold"]["language"] == language
        ][:limit // 2])
    seen = {row["id"] for row in selected}
    selected.extend(row for row in rows if row["id"] not in seen)
    return selected[:limit]

smoke_rows = smoke_rows_for_generation(validation_rows, limit=10)
smoke_predictions = []
for row in tqdm(smoke_rows, desc="Qwen adapter smoke generation", unit="doc"):
    smoke_predictions.append({"id": row["id"], "prediction": generate_json(row)})
smoke_metrics = benchmark_predictions(smoke_rows, smoke_predictions)
write_jsonl(output_dir / "smoke_predictions.jsonl", smoke_predictions)
(output_dir / "smoke_metrics.json").write_text(
    json.dumps(smoke_metrics, indent=2), encoding="utf-8"
)
print(json.dumps({
    "strictValidity": smoke_metrics["structuredOutputValidity"],
    "evidenceGrounding": smoke_metrics["evidenceGrounding"],
    "invalidOutputCount": smoke_metrics["invalidOutputCount"],
}, indent=2))
if smoke_metrics["structuredOutputValidity"] < 0.8:
    print(json.dumps(
        smoke_metrics["invalidOutputExamples"],
        indent=2,
        ensure_ascii=False,
    ))
    message = (
        "Qwen adapter failed the strict smoke gate. Treat this as a failed "
        "structured-output run; do not claim deployment readiness. Set "
        "MARKET_GYAN_ALLOW_FAILED_QWEN_SMOKE=true only if you need full-test "
        "diagnostics for a negative experiment."
    )
    if not ALLOW_FAILED_QWEN_SMOKE:
        raise RuntimeError(message)
    print(message)

## 7. Deterministic held-out generation with the adapter

In [ ]:
FastLanguageModel.for_inference(model)
adapter_predictions = []
for row in tqdm(test_rows, desc="Qwen adapter test generation", unit="doc"):
    adapter_predictions.append({"id": row["id"], "prediction": generate_json(row)})
write_jsonl(output_dir / "test_predictions.jsonl", adapter_predictions)

## 8. Score and plot all Qwen conditions

In [ ]:
import matplotlib.pyplot as plt
import re
import seaborn as sns
from market_gyan.dataset import (
    COMPACT_QWEN_FIELDS,
    CONFIDENCE_BANDS,
    EVENT_TYPES,
    IMPACT_DIRECTIONS,
    IMPACT_HORIZONS,
    IMPACT_MECHANISMS,
    IMPACT_SCOPES,
    NOT_RELEVANT_COMPACT_VALUES,
    RELEVANCE,
)
from market_gyan.metrics import benchmark_predictions

ENUMS = {
    "relevance": RELEVANCE,
    "eventType": EVENT_TYPES,
    "impactScope": IMPACT_SCOPES,
    "impactDirection": IMPACT_DIRECTIONS,
    "impactHorizon": IMPACT_HORIZONS,
    "impactMechanism": IMPACT_MECHANISMS,
    "confidenceBand": CONFIDENCE_BANDS,
}

ALIASES = {
    "relevant": "direct",
    "irrelevant": "not_relevant",
    "not relevant": "not_relevant",
    "not-relevant": "not_relevant",
    "positive": "bullish",
    "negative": "bearish",
    "mixed": "uncertain",
    "not applicable": "not_applicable",
    "not-applicable": "not_applicable",
    "short term": "short_term",
    "short-term": "short_term",
    "medium term": "medium_term",
    "medium-term": "medium_term",
    "ownership supply": "ownership_supply",
    "earnings cash flow": "earnings_cash_flow",
    "financing liquidity": "financing_liquidity",
    "market flow": "market_flow",
}

def normalize_enum(value, allowed):
    value = str(value).strip().strip('",.').lower()
    value = ALIASES.get(value, value)
    value = value.replace("-", "_").replace(" ", "_")
    return value if value in allowed else value

def parse_list_value(value):
    value = str(value).strip().rstrip(",")
    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except json.JSONDecodeError:
        pass
    bracket = re.search(r"\[(.*?)\]", value)
    if bracket:
        value = bracket.group(1)
    return [
        item.strip().strip('"\'')
        for item in re.split(r"[,;]", value)
        if item.strip().strip('"\'')
    ]

def repair_compact_prediction(prediction):
    if isinstance(prediction, dict) and "raw" not in prediction:
        return prediction
    raw = prediction.get("raw", "") if isinstance(prediction, dict) else str(prediction)
    repaired = {}
    for line in raw.splitlines():
        match = re.match(r'\s*[-*]?\s*`?"?([A-Za-z][A-Za-z0-9 _-]+)`?"?\s*[:=]\s*(.+?)\s*$', line)
        if not match:
            continue
        key = match.group(1).replace(" ", "").replace("-", "")
        field = next(
            (
                candidate for candidate in COMPACT_QWEN_FIELDS
                if key.lower() == candidate.lower()
            ),
            None,
        )
        if not field:
            continue
        value = match.group(2)
        if field in {"sectors", "symbols", "evidenceSentenceIds"}:
            repaired[field] = parse_list_value(value)
        elif field in ENUMS:
            repaired[field] = normalize_enum(value, ENUMS[field])
    if repaired.get("relevance") == "not_relevant":
        repaired.update(NOT_RELEVANT_COMPACT_VALUES)
    return repaired if repaired else prediction

benchmarks = {
    "zero_shot": benchmark_predictions(test_rows, zero_shot),
    "three_shot": benchmark_predictions(test_rows, three_shot),
    "unsloth_qlora": benchmark_predictions(test_rows, adapter_predictions),
}
if constrained_zero_shot:
    benchmarks["vllm_constrained_zero_shot"] = benchmark_predictions(
        test_rows,
        constrained_zero_shot,
    )
if constrained_three_shot:
    benchmarks["vllm_constrained_three_shot"] = benchmark_predictions(
        test_rows,
        constrained_three_shot,
    )
repaired_adapter_predictions = [
    {"id": row["id"], "prediction": repair_compact_prediction(row["prediction"])}
    for row in adapter_predictions
]
benchmarks["unsloth_qlora_repaired_diagnostic"] = benchmark_predictions(
    test_rows,
    repaired_adapter_predictions,
)
write_jsonl(
    output_dir / "qwen_repaired_diagnostic.jsonl",
    repaired_adapter_predictions,
)
(output_dir / "metrics.json").write_text(
    json.dumps(benchmarks, indent=2), encoding="utf-8"
)

labels = ["direct", "indirect", "not_relevant"]
matrix = [
    [benchmarks["unsloth_qlora"]["relevance"]["confusion"][actual][predicted]
     for predicted in labels]
    for actual in labels
]
quality_names = [
    "JSON validity", "grounding", "sector F1", "symbol F1", "evidence F1"
]
quality = [
    benchmarks["unsloth_qlora"]["structuredOutputValidity"],
    benchmarks["unsloth_qlora"]["evidenceGrounding"],
    benchmarks["unsloth_qlora"]["sectorMicroF1"],
    benchmarks["unsloth_qlora"]["symbolMicroF1"],
    benchmarks["unsloth_qlora"]["evidenceSentenceF1"],
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues", ax=axes[0],
    xticklabels=labels, yticklabels=labels
)
axes[0].set_title("Relevance confusion matrix")
benchmark_names = [
    name for name in (
        "zero_shot",
        "three_shot",
        "vllm_constrained_zero_shot",
        "vllm_constrained_three_shot",
        "unsloth_qlora",
    )
    if name in benchmarks
]
bars = axes[1].bar(
    benchmark_names,
    [benchmarks[name]["relevance"]["macroF1"] for name in benchmark_names],
)
axes[1].bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
axes[1].set_ylim(0, 1)
axes[1].set_title("Base versus Unsloth QLoRA relevance macro-F1")
bars = axes[2].barh(quality_names, quality)
axes[2].bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
axes[2].set_xlim(0, 1)
axes[2].set_title("Structured-output quality")
plt.tight_layout()
plt.savefig(output_dir / "test_results.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 9. Required ablations

Run a second training job with `include_hard_negatives=False` in
`make_dataset(...)` and compare relevance macro-F1. The RAG-enabled versus
RAG-disabled ablation is run through the system evaluation harness because
current factual knowledge must remain outside model weights.

In [ ]:
history = trainer.state.log_history
fig, axis = plt.subplots(figsize=(7, 4))
train_steps = [row["step"] for row in history if "loss" in row]
train_loss = [row["loss"] for row in history if "loss" in row]
eval_steps = [row["step"] for row in history if "eval_loss" in row]
eval_loss = [row["eval_loss"] for row in history if "eval_loss" in row]
if train_steps:
    axis.plot(train_steps, train_loss, label="train")
if eval_steps:
    axis.plot(eval_steps, eval_loss, label="validation")
axis.set_title("Qwen3-8B Unsloth QLoRA loss")
axis.legend()
plt.savefig(output_dir / "loss.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 10. Archive final adapter, predictions, metrics, and plots

In [ ]:
import shutil
archive = shutil.make_archive(str(output_dir), "zip", output_dir)
print(archive)
# Colab: from google.colab import files; files.download(archive)